<a href="https://colab.research.google.com/github/marisolriveraslrzn/CodingIA/blob/main/Text_generation_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text generation - Fine-tuning

1.   Trabajar con modelos de IA y con datasets de entrenamiento/prueba.

In [1]:
!pip install transformers datasets

2. Importa las herramientas:


In [3]:
# AutoTokenizer: traductor de texto a tokens (números).
# GPT2LMHeadModel: el “cerebro” GPT‑2 que predice palabras.
# pipeline: atajo para usar modelos sin tanto código.
# Trainer y TrainingArguments: piezas para entrenar el modelo.
# pandas y Dataset: para manejar datos como tablas.

from transformers import AutoTokenizer, GPT2LMHeadModel, pipeline, Trainer, TrainingArguments
import pandas as pd
from datasets import Dataset

#Define el modelo base que vamos a usar: GPT‑2.
model_name = "gpt2"

#Carga de tokenizador de GPT-2
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

#Carga modelo original, sin cambios
model_orig = GPT2LMHeadModel.from_pretrained(model_name)
model_orig.resize_token_embeddings(len(tokenizer))

#Carga de otro modelo, entrenado y con cambios
model_tuned = GPT2LMHeadModel.from_pretrained(model_name)
model_tuned.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Embedding(50257, 768)

3. Como el modelo GPT‑2 genera texto antes de ser entrenado -- es el “punto de partida” del modelo.


In [10]:
#Crea una “pipeline” (una línea de producción) que conecta el modelo y el tokenizador.
generator_orig = pipeline("text-generation", model=model_orig, tokenizer=tokenizer)

#prompt:es el texto inicial
prompt = "Game: Mario Bros\nDescription: "
result = generator_orig(
    prompt,
    max_new_tokens=150,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9)

print("Texto ANTES del entranamiento(training):")
print(result[0]["generated_text"])


[transformers] Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto ANTES del entranamiento(training):
Game: Mario Bros
Description: 他場斗限上の太郎を一合する他笑の方はある個に联芖されています。

他笑の方はある個に联芖されています。 You can also play in the "Koreatown" setting, a 3km ride away from the main hub at Chichi-cho!

You can also play in the "Koreatown" setting, a 3km ride away from the main hub atChichi-cho! You can also play in the "Koreatown" setting, a


4. Cargando datos y creando un conjunto de datos

In [11]:
#Abre el archivo de formato "txt" y lee todo su contenido.
with open("game.txt", "r", encoding="utf-8") as f:
    text = f.read()

texts = [blok.strip() for blok in text.split("===") if blok.strip()]

df = pd.DataFrame({"text": texts})
dataset = Dataset.from_pandas(df)


5. Tokenización

In [12]:
tokenizer.pad_token = tokenizer.eos_token

#Creamos una Función "tokenize": Convierte cada bloque de texto en tokens (números que representan palabras).
def tokenize(batch):
    tokens = tokenizer(batch["text"],
                       padding=True,    #rellena los textos cortos para que todos tengan la misma longitud.
                       truncation=True, #corta los textos largos para que no superen el límite.
                       max_length=512)  #fija el máximo de tokens por ejemplo.
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

#Aplica la función de tokenización a todo el dataset.
tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

6. Ajuste fino del modelo


In [13]:
#Ajuste fino del modelo

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=2,
    save_steps=1000,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model_tuned,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
2,7.282209
4,5.753758
6,5.122537
8,3.996161
10,4.353153
12,3.325219
14,3.788640
16,3.750498
18,3.604856
20,3.362799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=4.433982968330383, metrics={'train_runtime': 114.5064, 'train_samples_per_second': 0.437, 'train_steps_per_second': 0.175, 'total_flos': 1888243200000.0, 'train_loss': 4.433982968330383, 'epoch': 10.0})

7. Probamos el modelo después del entrenamiento

In [15]:
generator_tuned = pipeline("text-generation", model=model_tuned, tokenizer=tokenizer)

prompt = "Game: Stay\nDescription: "
result = generator_tuned(
    prompt,
    max_new_tokens=150,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9)

print("Texto DESPUES del entranamiento(training):")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto DESPUES del entranamiento(training):
Game: Stay
Description: --------------


In a post-apocalyptic world, an alien race of a strange group of aliens who seek to find their way back to their universe. They've been trapped in a mysterious world they have never seen before.


This is a game about exploration and war. Your goal is to find the life of the survivors from the old space opera, explore the world with your own story, and discover the world's secrets.


New player characters from the sci-fi game:

Infecting an alien race whose origins are hidden within a galaxy, the first game takes place in a dystopian, alien society known as The Deep.

With a unique game system, the player controls a new alien race through the alien's
